In [2]:
import h5py as h5
import torch
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt
from joblib import Parallel, delayed

In [3]:
data_path = '/home3/rfit/Telescope_Array/phd_work/data/normed/pr_photon_0001_excl_sat_T_excl_geo_T_one_work.h5'

In [31]:
'''
mc_params (num_evs,10):
0. mc_event_num
1. mc_parttype (CORSIKA, 1 - gamma, 14 - proton, 5626 - Fe)
2. mc_corecounter, closest to core detector number
3. mc_E (for primaries other than photon energy is rescaled by 1/1.27, i.e. to proton FD energy scale)
4. mc_theta
5. mc_phi
6. mc_height_1st_inter, km
7. mc_xcore
8. mc_ycore
9. mc_border_distance, km 
'''
with h5.File(data_path,'r') as f:
    print('keys', list(f.keys()))
    test = f['test']
    train = f['train']
    keys = list(test.keys())
    mc_params = test['mc_params'][:]
    for k in keys:
        print(k, test[k].shape)
    mc_params_train = train['mc_params'][:]
    recon_test = test['recos'][:]
    recon_train = train['recos'][:]
    


keys ['norm_param', 'test', 'train', 'val']
dt_mask (1143461, 2)
dt_params (14937993, 6)
ev_ids (1143461, 3)
ev_starts (1143462,)
mc_params (1143461, 10)
recos (1143461, 6)


In [7]:
cd ../src/train_VAE 

/home/rfit/Telescope_Array/phd_work/src/train_VAE


In [24]:
try:
    import importlib
    importlib.reload(pipline)
except NameError:
    import pipline
Pipeline = pipline.Pipline(config = 'config.yaml')

Using device: cuda
['catboost_info', 'pipline.py', '__pycache__', 'show_loss.py', 'model_old.py', 'metric.py', 'loss_hist.png', 'runs', 'test_runs', 'datasets.py', 'phd_work', 'runs_tests', 'model.py', 'loss.py', 'tests', 'utils.py', 'init.sh', 'config.yaml', '__init__.py', 'for_temporary_work', '.ipynb_checkpoints']
Encoder has params: 783944 Decoder has params: 2055719
split_num 10000 <class 'int'>
Saving Path: /home/rfit/Telescope_Array/phd_work/Models/AutoEncoder/info_Transfoemr_New_KL_without_MMD
self.koef_KL tensor([0.0000, 0.0034, 0.0069, 0.0103, 0.0138, 0.0172, 0.0207, 0.0241, 0.0276,
        0.0310, 0.0345, 0.0379, 0.0414, 0.0448, 0.0483, 0.0517, 0.0552, 0.0586,
        0.0621, 0.0655, 0.0690, 0.0724, 0.0759, 0.0793, 0.0828, 0.0862, 0.0897,
        0.0931, 0.0966, 0.1000], device='cuda:0', dtype=torch.float64)


In [25]:
model_path = '/home/rfit/Telescope_Array/phd_work/Models/AutoEncoder/info_Transfoemr_MMD_0.05_KL_0.01_new_MMD2_MMD_KL_0/best'
Pipeline.model.load(model_path)
Pipeline.model.eval()

load True format. If it is 'FALSE' we worning


VAE(
  (encoder): Encoder_Transformer(
    (embading): Linear(in_features=6, out_features=64, bias=True)
    (TransformerEncoderLayer): TransformerEncoderLayer(
      (self_attn): MultiheadAttention(
        (out_proj): NonDynamicallyQuantizableLinear(in_features=64, out_features=64, bias=True)
      )
      (linear1): Linear(in_features=64, out_features=1024, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (linear2): Linear(in_features=1024, out_features=64, bias=True)
      (norm1): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
      (norm2): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
      (dropout1): Dropout(p=0.1, inplace=False)
      (dropout2): Dropout(p=0.1, inplace=False)
    )
    (TransformerEncoder): TransformerEncoder(
      (layers): ModuleList(
        (0): TransformerEncoderLayer(
          (self_attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=64, out_features=64, bias=True)
          )
    

In [29]:
for x, part, params_CR in Pipeline.train_loader:
    x = x.to('cuda:0')
    latents = Pipeline.model.encoder(x)[0]
    print(latents.shape)
    print(params_CR) # MC en
    print(mc_params_train[:10,3])
    print('part', part)
    break

torch.Size([150, 8])
tensor([[ 66.6812],
        [ 17.9415],
        [ 81.4819],
        [ 55.1106],
        [ 45.3420],
        [ 43.3047],
        [ 12.8946],
        [100.7563],
        [166.2303],
        [195.0000],
        [106.3951],
        [ 28.7537],
        [289.3670],
        [ 20.9311],
        [194.6643],
        [ 59.5555],
        [320.6560],
        [  4.9834],
        [ 23.2209],
        [ 16.2484],
        [ 11.6439],
        [ 13.8191],
        [151.6672],
        [  4.2115],
        [ 24.4757],
        [ 58.7550],
        [322.8429],
        [142.2440],
        [ 88.4456],
        [  6.4501],
        [ 64.2493],
        [236.7699],
        [ 35.0818],
        [ 65.4586],
        [ 61.8143],
        [239.8180],
        [245.0370],
        [216.4588],
        [ 48.7683],
        [ 50.0873],
        [  7.7633],
        [  7.7509],
        [133.8420],
        [ 20.4473],
        [ 37.2137],
        [194.1780],
        [149.9400],
        [293.5237],
        [125.6011],

/home/rfit/Telescope_Array/phd_work/src/train_VAE/datasets.py:139: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  return torch.tensor(x), torch.tensor(mc_params[1]), torch.tensor(params_CR)


In [28]:
# то что выше показывает, что порядок сохранен

In [ ]:
all_information = None
start_index = 0
for x, part, params_CR in tqdm(Pipeline.train_loader):
    x = x.to('cuda:0')
    latents = Pipeline.model.encoder(x)[0]
    part = part.unsqueeze(1)
    part = part.to('cpu').detach().numpy()
    latents = latents.to('cpu').detach().numpy()
    recos = recon_train[start_index:start_index+x.shape[0]]
    start_index += x.shape[0]
    if all_information is None:
        all_information = np.concatenate([latents, recos, part], axis=1)
    else:
        all_information = np.concatenate([all_information, np.concatenate([latents, recos, part], axis=1)], axis=0)


  5%|▍         | 3279/68608 [00:54<21:23, 50.90it/s]

In [ ]:
np.save('all_information_train.npy', all_information)

In [ ]:
all_information_test = None
start_index = 0
for x, part, params_CR in tqdm(Pipeline.test_loader):
    x = x.to('cuda:0')
    latents = Pipeline.model.encoder(x)[0]
    part = part.unsqueeze(1)
    part = part.to('cpu').detach().numpy()
    latents = latents.to('cpu').detach().numpy()
    recos = recon_test[start_index:start_index+x.shape[0]]
    start_index += x.shape[0]
    if all_information_test is None:
        all_information_test = np.concatenate([latents, recos, part], axis=1)
    else:
        all_information = np.concatenate([all_information, np.concatenate([latents, recos, part], axis=1)], axis=0)